# Modeling

Packages and setup

In [ ]:
# Public packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import itertools
import math
import os
import re
from pathlib import Path
import tabulate
from IPython.display import display, Markdown
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import StandardScaler


# Set directory to project root
def find_project_root(start: Path = Path().absolute()) -> Path:
    for parent in start.parents:
        if (parent / "requirements.txt").exists(): return parent
    return start 
os.chdir(find_project_root())

# Preemptively set new Pandas option, also set matplotlib to close
pd.options.mode.copy_on_write = True
%matplotlib inline
%config InlineBackend.close_figures=True

# Allow reloading of custom Python classes without resetting kernel
pd.set_option('display.max_rows', 100)
%load_ext autoreload
%autoreload 2

Read data from parquet files

In [ ]:
# Load formatted data
%store -r static_data_merged
%store -r sales_data_merged

# Check if the data is already imported
if 'static_data_merged' not in locals():

    # Retrieve filnames
    static_data_filenames = os.listdir(f"2_palate_data_parquet_cleaned")

    # Initialize a dictionary for the four parquet files of "static reference data"
    static_data = {}
    for filename in tqdm(static_data_filenames):

        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/{filename}")
            base_name = re.sub(r'\.parquet$', '', filename)
            static_data[base_name] = df

    # Rename and store
    static_data_merged = static_data.copy()
    
# Data already exists
else:
    static_data = static_data_merged.copy()

%store static_data_merged

# Check if the data is already imported
if 'sales_data_merged' not in locals():

    # Retrieve filnames
    restaurant_sales_filenames = os.listdir(f"2_palate_data_parquet_cleaned/orders_item_level")

    # Initialize a dictionary for the 30 parquet files of "restaurant sales data"
    sales_and_menu_data = {}
    for filename in tqdm(restaurant_sales_filenames):
        
        # Exclude other files and folders
        if ".parquet" in filename:
            df = pd.read_parquet(f"2_palate_data_parquet_cleaned/orders_item_level/{filename}")
            location_id = re.sub(r'_sales_and_menu\.parquet$', '', filename)
            sales_and_menu_data[location_id] = df
    
    # Rename and store
    sales_data_merged = {}
    for loc_id, df in sales_and_menu_data.items():
        sales_data_merged[loc_id] = df.copy()
    
# Data already exists
else:
    sales_and_menu_data = {}
    for loc_id, df in sales_data_merged.items():
        sales_and_menu_data[loc_id] = df.copy()

%store sales_data_merged

# Rename data for ease of use
before_after_details = static_data['before_after_details'].copy() # Promotional items (30 rows)
customers = static_data['customers'].copy() # Specific customer information: for matching customers with orders
items_tagged = static_data['items_tagged'].copy() # Menu items for all restaurants: for matching plant-based labels with orders
locations = static_data['locations'].copy() # Restaurant details (30 rows)
location_ids = list(sales_and_menu_data.keys())

# True promos
%store -r before_after_details_true
if 'before_after_details_true' not in locals():
    before_after_details_true = pd.read_csv('data/before_after_details_true.csv', index_col='location_id')
%store before_after_details_true

# Timezones
%store -r timezones
if 'timezones' not in locals():
    timezones = pd.read_csv('data/timezones.csv', index_col='location_id')['timezone'].to_dict()
    for loc_id, df in sales_and_menu_data.items():
        df.index = df.index.tz_convert(timezones[loc_id])
        sales_and_menu_data[loc_id] = df
%store timezones

%store -r restaurants_by_4m_coverage
if 'restaurants_by_4m_coverage' not in locals():
    restaurants_by_4m_coverage = pd.read_csv('data/2_palate_data_parquet_cleaned/restaurants_by_4m_coverage.csv')['location_id'].tolist()
%store restaurants_by_4m_coverage

# Remove bad data
del sales_and_menu_data['AQD04SM0J92WA']
#
del sales_and_menu_data['LBMCPAYT7W36V']
del sales_and_menu_data['L3XS7WSJ4AJA3']
del sales_and_menu_data['1G5AJ17XCH2A8']
del sales_and_menu_data['3AXDVZJYN9DRS']
del sales_and_menu_data['MS8R16DY0JQAM']
#
del sales_and_menu_data['N0PC58FB2XAZ3']
del sales_and_menu_data['ADPFRN3QZRCXK']
del sales_and_menu_data['WJA3YCD4QBWRX']
del sales_and_menu_data['0RJH3FFPYBPEY']
del sales_and_menu_data['LZ5MR1TS37E7W']

location_ids = list(sales_and_menu_data.keys())
before_after_details = before_after_details.loc[restaurants_by_4m_coverage]
location_ids_by_coverage = pd.merge(pd.Series(np.zeros(30), index=restaurants_by_4m_coverage, name=1), pd.Series(np.zeros(19), index=location_ids, name=2), how='inner', left_index=True, right_index=True).index.tolist()

In [ ]:
locations.loc[location_ids_by_coverage,['city','state']]

In [ ]:
locations.loc[['EMBVNVD207CC6','C0BE4NDSW26QN','V3Q26BHF3SE2H','LBZEEFSBJNB3Z','SAFK7ND1HR6XS','S8MT0YGD2KTN9','LFZFT3VASXPED','1SQPTEGYPH0GA','9XKJD8DQTH559','LQ5EH4BKGV61T','78AY09MVJVTYE'],['city','state']]

Add promo exposure indicator

In [ ]:
# for loc_id, df in sales_menu_customers_data.items():
#     df = df.tz_localize(None)
#     df.loc[:,'promo'] = 0
#     df.loc[:before_after_details.loc[loc_id,'cross_over_date'], 'promo'] = 1

Putting in customer data

In [ ]:
# # Initialize dict all data
# sales_menu_customers_data = {}
# for loc_id, df in sales_and_menu_data.items():

#     # Prevent overwriting
#     df = df.copy()

#     # Keep the index, since merges don't keep it
#     df.reset_index(inplace=True)

#     # Double check customers are unique
#     customers.dropna(subset=['customer_id'], inplace=True)
#     customers.drop_duplicates(subset=['location_id', 'customer_id'], inplace=True)

#     # Combine
#     merged = pd.merge(df, customers, on=['location_id', 'customer_id'], how='left')

#     # Reset the index back to datetimes
#     merged.set_index('created_at', inplace=True, drop=False)

#     # Save
#     sales_menu_customers_data[loc_id] = merged

Roll data

Plotting tool

In [ ]:
# Plot function for resampling and visualization
def plot_resampled(data, freq, start_date=None, end_date=None, title="Resampled Predictions"):
    resampled_pred = data.resample(freq)['pred'].mean()
    resampled_actual = data.resample(freq)['vegan_outcome'].mean()

    if start_date and end_date:
        resampled_pred = resampled_pred.loc[start_date:end_date]
        resampled_actual = resampled_actual.loc[start_date:end_date]

    resampled_pred.plot(color='orange', label='Predicted', title=title)
    resampled_actual.plot(color='blue', alpha=0.3, label='Actual', title=title)

Logistic regression

In [ ]:
exogenous_predictors = [
    'meat_window_avg',
    #'vegetarian_window_avg',
    'vegan_window_avg',
]

time_predictors = [
    'hour_of_day',
    'meal_period',
    'weekend',
    'day_of_week',
    'day_of_month',
    'month',
    'season',
    'date',
]

predictors = exogenous_predictors + time_predictors

for i, loc_id in enumerate(location_ids_by_coverage):
    
    if i == 3 or i == 6 or i == 12:
        continue
    if i == 7:
        continue
    
    model_data_loc = model_data.query('location_id == @loc_id')

    # Split the data into training and testing sets
    train_size = model_data_loc.shape[0] // 2

    train_data = (model_data_loc
                  .dropna(subset=predictors)
                  .reset_index()
                  .iloc[:train_size, :])
    test_data = (model_data_loc
                 .dropna(subset=predictors)
                 .reset_index()
                 .iloc[train_size:, :])

    # Fit logistic regression model
    formula = ['vegan_outcome ~ ',
               'meat_window_avg + ',
               # 'vegetarian_window_avg',
               'vegan_window_avg + ',
               'vegan_window_avg:meat_window_avg + ',
               'hour_of_day + ',
               'meal_period + ',
               'weekend + ',
               'day_of_week + ',
               'day_of_month + ',
               'month + ',
               'season + ',
               'date'
            ]
    
    logit_model = smf.logit(''.join(formula), train_data)
    logit_fit = logit_model.fit(maxiter=200)

    train_data['pred'] = logit_fit.predict(train_data)
    train_data['residuals'] = train_data['vegan_outcome'] - train_data['pred']
    test_data['pred'] = logit_fit.predict(test_data)

    print(i+1)
    print(loc_id)
    
    train_result = train_data.set_index('created_at')
    test_result = test_data.set_index('created_at')

    # Resample and plot results for training data
    # plot_resampled(train_result, '7D', title="Training Data: Weekly Resampled Predictions")
    # plot_resampled(train_result, '7D', start_date='2020', end_date='2021', title="Training Data: Weekly (2020-2021)")
    # plot_resampled(train_result, '30D', title="Training Data: Monthly Resampled Predictions")
    # plt.show()

    # Resample and plot results for testing data
    plot_resampled(test_result, '7D', title="Testing Data: Weekly Resampled Predictions")
    # plot_resampled(test_result, '7D', start_date='2020', end_date='2021', title="Testing Data: Weekly (2020-2021)")
    # plot_resampled(test_result, '30D', title="Testing Data: Monthly Resampled Predictions")
    plt.show()

Negative Binomial Regression

In [ ]:
exogenous_predictors = [
    'meat_window_avg',
    'vegetarian_window_avg',
    'vegan_window_avg',
]

time_predictors = [
    'weekend',
    'day_of_week',
    'day_of_month',
    'month',
    'season',
    'date',
]

predictors = exogenous_predictors + time_predictors

for i, loc_id in enumerate(locations_id_by_coverage):
    
    if i == 3 or i == 12:
        continue
    
    model_data_loc = daily_model_data.query('location_id == @loc_id')

    # Split the data into training and testing sets
    train_size = model_data_loc.shape[0] // 2

    train_data = (model_data_loc
                  .dropna(subset=predictors)
                  .reset_index()
                  .iloc[:train_size, :])
    test_data = (model_data_loc
                 .dropna(subset=predictors)
                 .reset_index()
                 .iloc[train_size:, :])

    # Fit logistic regression model
    formula = ['vegan_outcome ~ ',
               'meat_window_avg + ',
               #'vegetarian_window_avg + ',
               'vegan_window_avg + ',
               'vegan_window_avg:meat_window_avg + ',
               'weekend + ',
               'day_of_week + ',
               'day_of_month + ',
               'month + ',
               'season'
               #'date'
            ]
    neg_binom_model = smf.negativebinomial(''.join(formula), 
                                           data=train_data)
    # neg_binom_model = cm.ZeroInflatedNegativeBinomialP.from_formula(''.join(formula),
    #                                                                 data=train_data,
    #                                                                 inflation="logit")
    neg_binom_fit = neg_binom_model.fit(maxiter=200)

    # Fit ARIMA model to residuals
    train_data['pred'] = neg_binom_fit.predict(train_data)
    # train_data['residuals'] = train_data['vegan_outcome'] - train_data['pred']
    # arima_model = ARIMA(train_data['residuals'], order=(1, 0, 1)).fit()
    # print(arima_model.summary())
    # print(neg_binom_fit.summary())

    # Predict for training and testing sets
    #train_data['arima_adjusted_pred'] = arima_model.fittedvalues + train_data['pred']
    test_data['pred'] = neg_binom_fit.predict(test_data)
    #test_data['arima_forecast'] = arima_model.get_forecast(steps=len(test_data)).predicted_mean
    #test_data['arima_adjusted_pred'] = test_data['pred'] + test_data['arima_forecast']

    print(i+1)
    print(loc_id)

    # Resample and plot results for training data
    train_result = train_data.set_index('created_at')
    #plot_resampled(train_result, '7D', title="Training Data: Weekly Resampled Predictions")
    #plot_resampled(train_result, '7D', start_date='2020', end_date='2021', title="Training Data: Weekly (2020-2021)")
    #plot_resampled(train_result, '30D', title="Training Data: Monthly Resampled Predictions")
    plt.show()

    # Resample and plot results for testing data
    test_result = test_data.set_index('created_at')
    plot_resampled(test_result, '7D', title="Testing Data: Weekly Resampled Predictions")
    #plot_resampled(test_result, '7D', start_date='2020', end_date='2021', title="Testing Data: Weekly (2020-2021)")
    #plot_resampled(test_result, '30D', title="Testing Data: Monthly Resampled Predictions")
    plt.show()

In [ ]:


# Function for resampling and visualization.
def plot_resampled(data, freq, start_date=None, end_date=None, title="Resampled Predictions"):
    resampled_pred = data.resample(freq)['pred'].mean()
    resampled_actual = data.resample(freq)['vegan_outcome'].mean()
    if start_date and end_date:
        resampled_pred = resampled_pred.loc[start_date:end_date]
        resampled_actual = resampled_actual.loc[start_date:end_date]
    resampled_pred.plot(color='orange', label='Predicted')
    resampled_actual.plot(color='blue', alpha=0.3, label='Actual')
    plt.title(title)
    plt.legend()

# List of candidate predictors.
candidate_predictors = ['meat_window_avg', 
                        'vegan_window_avg', 
                        'weekend',
                        'date', 
                        'day_of_week', 
                        #'day_of_month', 
                        'month', 
                        'season']

# Loop over locations (here, just the first one for demonstration).
for loc_id in locations_id_by_coverage[:1]:
    model_data_loc = daily_model_data.query('location_id == @loc_id')
    
    # Split the data into training and testing sets (first half for training).
    train_size = model_data_loc.shape[0] // 2
    train_data = model_data_loc.dropna(subset=candidate_predictors).reset_index().iloc[:train_size, :]
    test_data = model_data_loc.dropna(subset=candidate_predictors).reset_index().iloc[train_size:, :]
    
    best_model_error = np.inf  # Initialize best error to infinity.
    best_formula = None
    best_fit = None
    
    # Loop over all nonempty subsets of candidate predictors.
    for k in range(1, len(candidate_predictors) + 1):
        for subset in itertools.combinations(candidate_predictors, k):
            subset = list(subset)
            # Build the formula string.
            formula = "vegan_outcome ~ " + " + ".join(subset)
            # Optionally add an interaction if both predictors are present.
            if 'meat_window_avg' in subset and 'vegan_window_avg' in subset:
                formula += " + vegan_window_avg:meat_window_avg"
            try:
                # Fit a negative binomial model on training data.
                model = cm.ZeroInflatedNegativeBinomialP.from_formula(''.join(formula),
                                                                     data=train_data,
                                                                     inflation="logit")
                fit = model.fit(maxiter=200, disp=0)
                # Predict on held-out test data.
                test_pred = fit.predict(test_data)
                mse = np.mean((test_data["vegan_outcome"] - test_pred)**2)
                # Update best model if current MSE is lower.
                if mse < best_model_error:
                    best_model_error = mse
                    best_formula = formula
                    best_fit = fit
            except Exception as e:
                # Skip models that fail to fit.
                continue

    print("Best model for location", loc_id)
    print("Formula:", best_formula)
    print("Test MSE:", best_model_error)
    
    # Generate predictions for both training and testing data.
    train_data['pred'] = best_fit.predict(train_data)
    test_data['pred'] = best_fit.predict(test_data)
    
    # Resample and plot results for training data.
    train_result = train_data.set_index('created_at')
    plot_resampled(train_result, '7D', title="Training Data: Weekly Resampled Predictions")
    plt.show()
    
    # Resample and plot results for testing data.
    test_result = test_data.set_index('created_at')
    plot_resampled(test_result, '7D', title="Testing Data: Weekly Resampled Predictions")
    plt.show()
